# Bohemian matrices with population [0,1,2,...N], singular and non-singular distribution

The study of Bohemian matrices (acronym for Bounded Height Matrix of Integers) has emerged in the last decade as a fertile field at the intersection of numerical linear algebra, combinatorics, and number theory. While random matrices with continuous entries (Gaussian, for example) have been exhaustively studied since the works of Wigner and Dyson, matrices with discrete and bounded entries present unique spectral and structural behaviors that challenge classical intuition. In the continuum, the probability of finding a singular matrix is zero; in the discrete Bohemian domain, singularity is an event of finite and structured probability, whose distribution holds profound information about the arithmetic nature of linear transformations.

The present report aims to dissect the methodology for the analysis of a specific family of Bohemian matrices: those square matrices of dimension $N \times N$ whose entries belong to the population $P_N = \{0, 1, 2, \dots, N\}$. If one were to establish $N=q-1$, where $q$ is a prime number, this could be related to the field of lattice-based cryptography. Unlike the commonly studied $\{0, 1\}$ or ternary $\{-1, 0, 1\}$ populations, the population $P_N$ scales with the matrix dimension, introducing a factorial complexity in the search space $\Omega_N$, whose cardinality $|\Omega_N| = (N+1)^{N^2}$ grows at superexponential rates. The progress presented in this document shows the distribution of non-singular and singular matrices, the types of singular ones, and manages to generalize on certain aspects of combinatorics related to singularities of a structural character. 

This document serves as a living log of the research on the distribution of determinants and the singularity density in matrices with bounded discrete entries (Bohemian).

**Objectives:**

* Numerical Validation: Use Monte Carlo methods and parallelized brute force to obtain empirical statistics.

* Analytical Formalization: Deduce closed formulas based on combinatorics (Inclusion-Exclusion Principle and Bell Numbers) to categorize singularity structures (zero rows, zero columns, identical rows) without the need to generate them.

Each section comes with code accompanied by an explanation associated with the methodology and a brief analysis of the results obtained.

## Initial Stochastic Approximation: Parallelized Monte Carlo

Before addressing the analytical complexity, we establish a numerical baseline. For large dimensions $N$, the space $\Omega_N$ is intractable ($N=5 \implies 6^{25}$ matrices). Therefore, we implement a massive Monte Carlo generator.
### Implementation

The following code implements a Map-Reduce scheme using joblib.
* Memory Management: We do not generate giant lists. We process by "batches" (BATCH_SIZE) that fit comfortably in the processor's L3 cache.
* Vectorization: Instead of native Python loops, we generate $(B, N, N)$ tensors and use NumPy's optimized LAPACK routines (np.linalg.det) to calculate $100,000$ determinants simultaneously.
* Numerical Stability: Given that we work with integers, any determinant like 3.00000000004 is a floating-point error. We apply rounding and sign normalization (-0.0 to 0.0) to ensure the uniqueness of keys in the histogram.

In [4]:
import numpy as np
import pandas as pd
from collections import Counter
from joblib import Parallel, delayed
import time
import os

# --- GLOBAL PARAMETER CONFIGURATION ---
# Simulation magnitude and memory limits definition
TOTAL_SAMPLES = 1_000_000_000  # Target: 1 Billion matrices (10^9)
SAFE_BATCH = 10_000_000        # Buffer size to avoid RAM saturation

def simulate_chunk(amount, n_dim):
    """
    Simulation core: Generates a subset of matrices and calculates their determinants.
    
    Args:
        amount (int): Number of matrices to generate in this thread.
        n_dim (int): Dimension N of the square matrix.
        
    Returns:
        dict: Compressed dictionary {determinant_value: frequency}.
    """
    # Vectorized generation of random matrices with entries {0,..., n_dim}
    matrices = np.random.randint(0, n_dim + 1, size=(amount, n_dim, n_dim), dtype=np.int8)
    
    # Numerical calculation of the determinant using LU decomposition (numpy backend)
    dets = np.linalg.det(matrices)
    
    # Floating point error correction
    dets = np.round(dets, 1)
    dets[dets == -0.0] = 0.0  # Negative zero normalization
    
    # Frequency counting (partial histogram)
    unique, counts = np.unique(dets, return_counts=True)
    return dict(zip(unique, counts))

def run_giant_monte_carlo(n, writer, start_col):
    """
    Parallel process orchestrator. Manages work division into batches
    and consolidates progressive results.
    """
    print(f"\n--- STARTING SIMULATION FOR N={n} :: TARGET: {TOTAL_SAMPLES:,} SAMPLES ---")
    
    total_counts = Counter()
    current_processed = 0
    next_milestone = 10  # Initial percentage for progress reporting
    
    # Hardware resource detection
    n_cores = os.cpu_count()
    
    # --- MAIN PROCESSING LOOP ---
    while current_processed < TOTAL_SAMPLES:
        # 1. Calculate current batch size (limited by SAFE_BATCH or remainder)
        remaining = TOTAL_SAMPLES - current_processed
        current_step = min(remaining, SAFE_BATCH)
        
        # 2. Load distribution among cores (Load Balancing)
        chunk_per_core = current_step // n_cores
        chunks = [chunk_per_core] * n_cores
        
        # Residual adjustment if division is not exact
        if sum(chunks) < current_step:
            chunks[-1] += (current_step - sum(chunks))
            
        # 3. Parallel Execution (Implicit Map-Reduce Model)
        results = Parallel(n_jobs=-1)(
            delayed(simulate_chunk)(c, n) for c in chunks
        )
        
        # 4. Partial results consolidation
        for res in results:
            total_counts.update(res)
            
        current_processed += current_step
        
        # --- STATUS REPORT ---
        current_percentage = (current_processed / TOTAL_SAMPLES) * 100
        if current_percentage >= next_milestone:
            print(f"    -> Progress: {int(current_percentage)}% completed...")
            next_milestone += 10

    # --- RESULTS EXPORT ---
    print("Consolidating final data and exporting to Excel...")
    col_name_det = f'Det (N={n})'
    col_name_freq = f'Freq (1B)'
    
    # Results DataFrame creation
    df = pd.DataFrame(list(total_counts.items()), columns=[col_name_det, col_name_freq])
    
    # Empirical probability calculation
    df['Probability (%)'] = (df[col_name_freq] / TOTAL_SAMPLES) * 100
    df = df.sort_values(by=col_name_det)
    
    # Spreadsheet writing
    df.to_excel(writer, index=False, startcol=start_col)
    print(f"Simulation for N={n} finished successfully.")
    
    return start_col + 4

# --- MAIN EXECUTION BLOCK ---
if __name__ == '__main__':
    # Study range definition: N=1, 2, 3 (Implicit Brute Force) and N=4 (Laplace/Monte Carlo), and every other dimension has no viable method besides Monte Carlo
    vals = 6
    N_TO_SIMULATE = np.arange(1, vals + 1, 1)
    
    print(f"Starting simulation sequence for dimensions: {N_TO_SIMULATE}")
    filename = 'determinants_1B_safe.xlsx'
    col = 0
    
    tic = time.perf_counter()
    
    # Excel writer initialization (openpyxl engine)
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        for n in N_TO_SIMULATE:
            # Iterative execution by dimension
            col = run_giant_monte_carlo(n, writer, col)
            
    toc = time.perf_counter()
    
    # Final time metrics report
    print(f"\n=== TOTAL EXECUTION TIME: {toc-tic:.2f} s ({ (toc-tic)/60:.1f} min) ===")

Starting simulation sequence for dimensions: [1 2 3 4 5 6]

--- STARTING SIMULATION FOR N=1 :: TARGET: 1,000,000,000 SAMPLES ---
    -> Progress: 10% completed...
    -> Progress: 20% completed...
    -> Progress: 30% completed...
    -> Progress: 40% completed...
    -> Progress: 50% completed...
    -> Progress: 60% completed...
    -> Progress: 70% completed...
    -> Progress: 80% completed...
    -> Progress: 90% completed...
    -> Progress: 100% completed...
Consolidating final data and exporting to Excel...
Simulation for N=1 finished successfully.

--- STARTING SIMULATION FOR N=2 :: TARGET: 1,000,000,000 SAMPLES ---
    -> Progress: 10% completed...
    -> Progress: 20% completed...
    -> Progress: 30% completed...
    -> Progress: 40% completed...
    -> Progress: 50% completed...
    -> Progress: 60% completed...
    -> Progress: 70% completed...
    -> Progress: 80% completed...
    -> Progress: 90% completed...
    -> Progress: 100% completed...
Consolidating final data a

It is worth noting how 1 billion simulations were arbitrarily established, which for the first three dimensions is unnecessary, since they possess fewer matrices than the sample taken. However, it becomes more relevant starting from the fourth. The Excel file generates the raw data, but by graphing it, one can observe a constancy in the probability of finding any determinant value, with an increase in probability as it approaches zero. For the first dimensions, the zero determinant is a substantial portion of the possible values, but as dimensions increase, a rapid decrease in the probability of a singularity is perceived. This can be related to previous studies performed (as this is a preliminary work used as a logbook, sources are not available within the document; that would be done within a LaTeX file in the future, along with the following advances desired regarding eigenvalues, the trace, among others).

Tao and Vu's theorems on the singularity of discrete random matrices focus mainly on fixed and small alphabets (e.g., Bernoulli matrices with $\pm 1$ entries), demonstrating that the probability of singularity decays exponentially ($P_{sing} \le c^N$). Although the matrices studied in this work possess a growing alphabet $\mathcal{A}_N = \{0, \dots, N\}$, Tao and Vu's theorems are applicable as a conservative upper bound. The universality principle suggests that if singularity is asymptotically rare for restricted alphabets, it is even more so for alphabets whose cardinality grows with $N$, where the entropy of the rows is higher and the probability of linear collision decreases drastically.

## Categorización Estructural Estricta (N $\leq$ 3)

For small dimensions ($N=1, 2, 3$), we can contrast the previous simulation with the "Ground Truth" obtained by exhaustive brute force.

To simplify the analysis below, the following abbreviations are arbitrarily established to qualify the types of matrices that can be found:
1. **SM (Same)**: Trivial matrices formed by a single repeated scalar.
2. **ZR (Zero Row)**: There is a row of zeros.
3. **ZC (Zero Column)**: There is a column of zeros.
4. **ZRC (Zero Row and Column)**: There is a row and a column of zeros.
5. **ID (Identical)**: There is some repeated row.
6. **LD (Linear Dependence)**: The residue, those dense matrices that are in turn singular due to complex arithmetic dependence.
7. **NON (Non-singular)**: Matrices whose determinant is different from zero. They are part of the dense matrices.

It is of utmost importance to mention that SM, ZR, ZC, ZRC, ID possess structural singularities, while a visual, geometric, or similar argument cannot be made for the LD types. The LD and NON matrices make up the dense matrices. As will be seen in a later section, this allows generalizing via combinatorics the number of singular matrices by structure that will be obtained by dimension, and it will be possible to deduce what the sum of LD and NON will be, but currently it is not possible to determine exactly how many of each will be obtained without performing the exhaustive calculation beforehand. However, for the first three dimensions, this exhaustive calculation is quick to perform, which is why it is computed directly.

In [ ]:
import numpy as np
import itertools
import time

# =============================================================================
# VISUAL REPORT
# =============================================================================
def print_report(N, total, sm, zrc, zr, zc, id_count, ld, non, elapsed):
    print(f"\n{'='*60}")
    print(f" PROCESSING N={N} (PURE BRUTE FORCE - No Formulas)")
    print(f"{'='*60}")
    print(f" Time: {elapsed:.4f}s | Total Space: {total:,}")
    print(f"{'-'*60}")
    print(f" SM  (Scalar Singular):     {sm:>15,}")
    print(f" ZRC (Row & Col Zero):      {zrc:>15,}")
    print(f" ZR  (Row Zero Only):       {zr:>15,}")
    print(f" ZC  (Col Zero Only):       {zc:>15,}")
    print(f" ID  (Identical Rows):      {id_count:>15,}")
    print(f" LD  (Linear Dependence):   {ld:>15,}")
    print(f" NON (Non-Singular):        {non:>15,}")
    print(f"{'-'*60}")
    
    total_sum = sm + zrc + zr + zc + id_count + ld + non
    check = "OK" if total_sum == total else f"ERROR (Diff: {total - total_sum})"
    print(f" TOTAL BALANCE:             {total_sum:>15,}  [{check}]")

# =============================================================================
# ITERATIVE CLASSIFICATION LOGIC
# =============================================================================
def solve_bruteforce_pure(N):
    v = N + 1
    total_matrices = v**(N*N)
    
    # Counters
    c_sm = 0
    c_zrc = 0
    c_zr = 0
    c_zc = 0
    c_id = 0
    c_ld = 0
    c_non = 0
    
    # Pre-compute zero tuple for fast comparison
    zero_row_tuple = tuple([0] * N)
    
    tic = time.perf_counter()
    
    # --- MAIN LOOP: ITERATE OVER EVERY POSSIBLE MATRIX ---
    # itertools.product generates all combinations of 0..v-1
    for flat_m in itertools.product(range(v), repeat=N*N):
        
        # 1. Construct matrix and calculate determinant
        arr = np.array(flat_m, dtype=np.int8).reshape(N, N)
        det = np.linalg.det(arr)
        
        # 2. Verify Singularity
        # We use tolerance for floating point
        if abs(det) > 1e-9:
            c_non += 1
            continue # If not singular, skip to the next
            
        # --- If we get here, IT IS SINGULAR (Det == 0). We classify: ---
        
        # A. Check SM (Scalar Matrix)
        # All elements are equal to the first one
        if all(x == flat_m[0] for x in flat_m):
            c_sm += 1
            continue
            
        # B. Check Zeros
        # Convert to tuples for fast searches
        rows = [tuple(r) for r in arr]
        cols = [tuple(c) for c in arr.T]
        
        has_zr = zero_row_tuple in rows
        has_zc = zero_row_tuple in cols
        
        if has_zr and has_zc:
            c_zrc += 1
            continue
        elif has_zr:
            c_zr += 1
            continue
        elif has_zc:
            c_zc += 1
            continue
            
        # C. Check ID (Identical Rows)
        # We only check rows, just as in your original logic.
        # If the number of unique rows is less than N, there are repeated ones.
        if len(set(rows)) < N:
            c_id += 1
            continue
            
        # D. LD (Linear Dependence)
        # If it is singular and did not fall into any previous category, it is pure LD.
        c_ld += 1

    elapsed = time.perf_counter() - tic
    print_report(N, total_matrices, c_sm, c_zrc, c_zr, c_zc, c_id, c_ld, c_non, elapsed)

if __name__ == "__main__":
    # Execute for N=1, 2, 3
    # NOTE: N=3 takes a few seconds because it analyzes 262,144 matrices one by one.
    for n in range(1, 4):
        solve_bruteforce_pure(n)

## Categorización Estructural Estricta (N = 4)

Para $N=4$, el espacio es de $1.52 \times 10^{11}$ matrices. La fuerza bruta pura toma en este computador alrededor de 33 horas, o aun menos con un computador con más núcleos. Sin embargo, esto es vastamente ineficiente, entonces se realiza lo siguiente:

**Expansión de Cofactores**
En lugar de generar matrices completas $4 \times 4$, generamos geometrías base de 3 filas ($N \times (N-1)$).

* Pre-cálculo de Cofactores: Para cada conjunto de 3 filas, calculamos los 4 cofactores que resultan de expandir el determinante a lo largo de la (hipotética) cuarta fila. Esto reduce el cálculo del determinante a un producto punto:$$\det(A) = \vec{r}_4 \cdot \vec{C}_{base}$$

* Vectorización Masiva: Multiplicamos el vector de cofactores $\vec{C}_{base}$ contra todas las posibles filas cuartas ($\vec{r}_4 \in P^4$) en una sola operación matricial.

* Generadores Lazy: Usamos itertools.combinations_with_replacement sobre las filas únicas. Esto comprime el espacio de búsqueda significativamente porque no nos importa el orden de las filas para calcular la singularidad (solo sus valores), y luego ajustamos los conteos usando pesos multinomiales.

In [7]:
import numpy as np
import itertools
import math
import time
from collections import Counter
from joblib import Parallel, delayed

# --- CONFIGURACIÓN ---
N = 4
V = 5  # Valores 0..4

def get_unique_permutations_count(indices):
    """
    Versión optimizada de conteo de permutaciones para tupla de 4 elementos.
    Evita overhead de Counter.
    """
    # Ordenamos para comparar
    a, b, c, d = indices # Ya vienen ordenados r1<=r2<=r3<=r4 casi siempre
    
    # Lógica hardcodeada para N=4 es más rápida que loops genéricos
    if a == d: return 1           # (x,x,x,x) -> 1
    if a == c or b == d: return 4 # (x,x,x,y) o (x,y,y,y) -> 4
    if a == b and c == d: return 6 # (x,x,y,y) -> 6
    if a == b or b == c or c == d: return 12 # (x,x,y,z), (x,y,y,z), (x,y,z,z) -> 12
    return 24 # (x,y,z,w) -> 24

def procesar_chunk_turbo(chunk_indices, all_rows, row_masks, row_is_uniform, idx_zero_row):
    """
    Chunk optimizado con operaciones de bits y sin creación de arrays numpy.
    """
    local_counts = np.zeros(7, dtype=np.int64) 
    # Indices: 0:SM, 1:ZRC, 2:ZR, 3:ZC, 4:ID, 5:LD, 6:NON (no usado aquí)
    
    for idxs_3 in chunk_indices:
        r1, r2, r3 = idxs_3
        
        # 1. Construir matriz base 3x4 (Solo aquí usamos numpy)
        # Esto es inevitable para el determinante, pero es vectorizado
        block_3 = all_rows[[r1, r2, r3]]
        
        # 2. Calcular Cofactores (Laplace)
        # Optimizamos extrayendo columnas directamente
        c0 = block_3[:, 0]
        c1 = block_3[:, 1]
        c2 = block_3[:, 2]
        c3 = block_3[:, 3]
        
        # Determinantes 3x3 hardcodeados (Más rápido que np.linalg.det repetido)
        # Cofactor 0 (cols 1,2,3)
        m0 = c1[0]*(c2[1]*c3[2] - c2[2]*c3[1]) - c1[1]*(c2[0]*c3[2] - c2[2]*c3[0]) + c1[2]*(c2[0]*c3[1] - c2[1]*c3[0])
        # Cofactor 1 (cols 0,2,3)
        m1 = c0[0]*(c2[1]*c3[2] - c2[2]*c3[1]) - c0[1]*(c2[0]*c3[2] - c2[2]*c3[0]) + c0[2]*(c2[0]*c3[1] - c2[1]*c3[0])
        # Cofactor 2 (cols 0,1,3)
        m2 = c0[0]*(c1[1]*c3[2] - c1[2]*c3[1]) - c0[1]*(c1[0]*c3[2] - c1[2]*c3[0]) + c0[2]*(c1[0]*c3[1] - c1[1]*c3[0])
        # Cofactor 3 (cols 0,1,2)
        m3 = c0[0]*(c1[1]*c2[2] - c1[2]*c2[1]) - c0[1]*(c1[0]*c2[2] - c1[2]*c2[0]) + c0[2]*(c1[0]*c2[1] - c1[1]*c2[0])

        # Vector Normal C = [m0, -m1, m2, -m3]
        # Producto punto masivo: all_rows @ C
        # Expandimos: r[:,0]*m0 - r[:,1]*m1 + r[:,2]*m2 - r[:,3]*m3
        dots = all_rows[:,0]*m0 - all_rows[:,1]*m1 + all_rows[:,2]*m2 - all_rows[:,3]*m3
        
        # Filtrar candidatos (Producto punto == 0)
        valid_r4 = np.where(dots == 0)[0]
        
        # Filtrar por orden canónico (r4 >= r3)
        # Esto reduce drásticamente el bucle siguiente
        valid_r4 = valid_r4[valid_r4 >= r3]
        
        # --- BUCLE INTERNO ULTRA RÁPIDO ---
        # Pre-cargamos propiedades de las filas fijas
        mask_123 = row_masks[r1] & row_masks[r2] & row_masks[r3]
        has_zr_123 = (r1 == idx_zero_row) or (r2 == idx_zero_row) or (r3 == idx_zero_row)
        
        for r4 in valid_r4:
            # Peso combinatorio
            weight = get_unique_permutations_count((r1, r2, r3, r4))
            
            # --- CLASIFICACIÓN JERÁRQUICA (Sin NumPy) ---
            
            # 1. SM (Scalar Matrix)
            # Solo si r1=r2=r3=r4 y además es fila uniforme
            if r1 == r4: # Implica r1=r2=r3=r4 porque r1<=r2<=r3<=r4
                if row_is_uniform[r1]:
                    local_counts[0] += weight # SM
                    continue

            # 2. ZRC / ZR / ZC
            # Bitwise check para columnas cero
            has_zc = (mask_123 & row_masks[r4]) > 0
            has_zr = has_zr_123 or (r4 == idx_zero_row)
            
            if has_zr and has_zc:
                local_counts[1] += weight # ZRC
                continue
            if has_zr:
                local_counts[2] += weight # ZR
                continue
            if has_zc:
                local_counts[3] += weight # ZC
                continue
                
            # 3. ID (Identical Rows)
            # Si hay repetidos en (r1, r2, r3, r4).
            # Como están ordenados, solo chequeamos vecinos
            if r1 == r2 or r2 == r3 or r3 == r4:
                local_counts[4] += weight # ID
                continue
                
            # 4. LD (Linear Dependence)
            local_counts[5] += weight # LD

    return local_counts

def ejecutar_turbo_n4():
    print(f"\n{'='*60}")
    print(f" MODO TURBO N=4 (Validación Numérica)")
    print(f"{'='*60}")
    
    tic = time.perf_counter()
    
    # 1. Generar Universo y Pre-cálculos
    vals = np.arange(V, dtype=np.int8)
    all_rows = np.array(list(itertools.product(vals, repeat=N)), dtype=np.int64) # int64 para evitar overflow en dots
    n_rows = len(all_rows)
    
    # A. Máscaras de Ceros (Bitwise)
    # [0, 2, 0, 1] -> 1010 -> Si hay un cero, ponemos 1 en la mascara para AND
    # Espera, para ZC necesitamos que la columna SEA cero.
    # Máscara: 1 si es cero, 0 si tiene valor.
    # Row: [0, 2, 0, 1] -> Mask: [1, 0, 1, 0] (binario 10)
    row_masks = np.zeros(n_rows, dtype=np.int32)
    for i in range(n_rows):
        mask = 0
        for j in range(N):
            if all_rows[i, j] == 0:
                mask |= (1 << j)
        row_masks[i] = mask
        
    # B. Uniformidad
    row_is_uniform = np.array([len(set(row)) == 1 for row in all_rows], dtype=bool)
    
    # C. Índice Fila Cero
    idx_zero_row = -1
    for i in range(n_rows):
        if np.all(all_rows[i] == 0):
            idx_zero_row = i
            break
            
    print(f" Universo preparado. Filas: {n_rows}")
    
    # 2. Generador de Combinaciones
    # Reducimos un poco el Batch Size para ver actualizaciones más seguido
    comb_iter = itertools.combinations_with_replacement(range(n_rows), 3)
    BATCH_SIZE = 50_000 
    
    def chunk_generator():
        while True:
            chunk = tuple(itertools.islice(comb_iter, BATCH_SIZE))
            if not chunk: break
            yield chunk
            
    # 3. Ejecución Paralela
    print(" Iniciando Turbo-Scan...")
    
    # verbose=10 imprime mucho, usaremos verbose=5
    results = Parallel(n_jobs=-1, verbose=5)(
        delayed(procesar_chunk_turbo)(chunk, all_rows, row_masks, row_is_uniform, idx_zero_row) 
        for chunk in chunk_generator()
    )
    
    # 4. Consolidación
    final_arr = np.sum(results, axis=0)
    cats = ['SM', 'ZRC', 'ZR', 'ZC', 'ID', 'LD']
    final_counts = dict(zip(cats, final_arr))
    
    elapsed = time.perf_counter() - tic
    
    # 5. Reporte
    total_space = V**(N*N)
    total_singular = sum(final_counts.values())
    final_counts['NON'] = total_space - total_singular
    
    print(f"\n{'='*60}")
    print(f" RESULTADOS FINALES TURBO N=4")
    print(f" Tiempo: {elapsed/60:.2f} minutos")
    print(f"{'-'*60}")
    
    orden = ['SM', 'ZRC', 'ZR', 'ZC', 'ID', 'LD', 'NON']
    suma_check = 0
    
    for cat in orden:
        val = final_counts[cat]
        suma_check += val
        pct = (val / total_space) * 100
        print(f" {cat:<5}: {val:>18,}  ({pct:.6f}%)")
        
    print(f"{'-'*60}")
    print(f" TOTAL : {suma_check:>18,}")
    print(f" ESPACIO: {total_space:>18,}")
    
    if total_space == suma_check:
        print(" [OK] BALANCE PERFECTO")
    else:
        print(f" [!] DIFERENCIA: {total_space - suma_check}")

if __name__ == '__main__':
    ejecutar_turbo_n4()


 MODO TURBO N=4 (Validación Numérica)
 Universo preparado. Filas: 625
 Iniciando Turbo-Scan...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   21.6s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:  2.4min
[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:  3.9min
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed:  5.8min
[Parallel(n_jobs=-1)]: Done 640 tasks      | elapsed:  7.9min



 RESULTADOS FINALES TURBO N=4
 Tiempo: 9.81 minutos
------------------------------------------------------------
 SM   :                  5  (0.000000%)
 ZRC  :         30,525,376  (0.020005%)
 ZR   :        943,695,872  (0.618461%)
 ZC   :        943,695,872  (0.618461%)
 ID   :      1,408,918,524  (0.923349%)
 LD   :      5,963,540,208  (3.908266%)
 NON  :    143,297,514,768  (93.911459%)
------------------------------------------------------------
 TOTAL :    152,587,890,625
 ESPACIO:    152,587,890,625
 [OK] BALANCE PERFECTO


[Parallel(n_jobs=-1)]: Done 818 out of 818 | elapsed:  9.8min finished


## Generalización por combinatoria para singularidades estructurales

Concisamente, se utiliza teoría de conjuntos y el principio de Inclusión-Exclusión para obtener fórmulas que generalicen la cantidad de singularidades estructurales a gran detalle, dejando LD y NON dentro de un grupo que actualmente no se puede separar. Para esta sección, se define formalmente $v=N+1$, tal que la cardinalidad para una dimensión $N$ es $|\Omega|=T=v^{N^2}$

### 1. **Matrices Densas ($K_{dense}$)**: 
Es la cardinalidad del conjunto de matrices densas estructurales ($\mathcal{D}$) que no tienen ninguna fila nula ni ninguna columna nula. Para calcularlo, fijamos primero el universo de matrices que no tienen filas nulas: $(v^N - 1)^N$ (ya que cada una de las $N$ filas debe ser un vector no nulo). Sobre este universo, aplicamos PIE para excluir las matrices con columnas nulas.

$$\mathcal{D} = A^c \cap B^c$$

Siendo $B$ el evento describiendo cuando se tiene al menos una columna nula. Sea $P_j$ la propiedad de que la columna $j$ es nula. Buscamos el número de elementos en $A^c$ que no satisfacen ninguna $P_j$.

$$K = \sum_{k=0}^N (-1)^k S_k$$

Donde $S_k$ es la suma de los tamaños de las intersecciones de $k$ columnas fijadas a cero. Si se fijan $k$ columnas específicas a cero:

1. **Restricción de Columnas**: Estas $k$ columnas son obligatoriamente $\vec{0}$

2. **Grados de Libertad**: Nos quedan $N-k$ columnas libres

3. **Restricción de Filas ($A^c$)**: La matriz resultante no puede tener filas nulas
Al fijar $k$ columnas a cero, cada fila de la matriz se convierte efectivamente en un vector de dimensión $N-k$. Para que la fila original no sea nula, este sub-vector de dimensión $N-k$ no puede ser todo ceros. Entonces, el número de vectores válidos de longitud $N-k$ es: $v^{N-k} - 1$.
4. **Independencia**: Como elegimos las $N$ filas independientemente:
$$\text{Espacio válido dado } k \text{ columnas nulas} = (v^{N-k} - 1)^N$$
5. Multiplicando por las formas de elegir las columnas $\binom{N}{k}$ y el signo alternante, se obtiene lo siguiente:
   $$K = \sum_{k=0}^N (-1)^k \binom{N}{k} (v^{N-k} - 1)^N$$

### 2. Matrices con singularidades estructurales involucrando el cero

Se deduce que la unión de singularidades estructurales es el complemento de las densas:
$$|\mathcal{R} \cup \mathcal{C}| = T - K$$

Queremos la intersección $I_{rc} = |\mathcal{R} \cap \mathcal{C}|$ (matrices con filas Y columnas nulas). Por la identidad fundamental de conjuntos $|A \cup B| = |A| + |B| - |A \cap B|$, despejamos la intersección:
$$|\mathcal{R} \cap \mathcal{C}| = |\mathcal{R}| + |\mathcal{C}| - |\mathcal{R} \cup \mathcal{C}|$$

Sustituyendo (y usando simetría $|\mathcal{R}| = |\mathcal{C}| = U_r$):

$$I_{rc} = 2 U_r - (T - K)$$

Con todos estos componentes, se tiene suficiente información para obtener los tipos ZR,ZC y ZRC.

1. **SM**: Es el caso más trivial
$$\text{SM}=v$$
3. **ZR**: Matrices con filas nulas que no tienen columnas nulas
   $$ZR = U_r - I_{rc}$$
5. **ZC**: Matrices con columnas nulas que no tienen filas nulas
   $$ZC = U_c - I_{rc}$$
7. **ZRC**:Es la intersección pura, pero restando 1 (Con tal de no contar las matrices SM dos veces)
   $$ZRC = I_{rc} - 1$$


Ahora, es importante considerar lo siguiente para los cálculos en sí:
* Debido a la simetría de las matrices, $U_r=U_c$
* $U_r =U_c= T - (v^N - 1)^N$
* $I_{rc} = 2 U_r - T + K$



### 3. Singularidades de matrices con filas identicas (ID)
La fórmula para ID no es trivial porque no podemos simplemente contar matrices con filas repetidas; debemos contar solo aquellas que son densas (sin ceros) y que no son escalares (SM). Para lograr esto, mapeamos el problema a particiones de conjuntos.

1. **Particiones (Números de Bell)**: Las filas repetidas definen una relación de equivalencia entre los índices de las filas. Si la fila 1 es igual a la fila 2, los índices $\{1, 2\}$ están en el mismo bloque. Iteramos sobre todas las particiones $\pi$ del conjunto $\{1, \dots, N\}$.
2. **Coeficiente de Möbius:** Usamos un coeficiente $\mu(\pi)$ derivado del retículo de particiones para corregir el sobreconteo inherente (ej. si fila 1=2=3, esto se cuenta en 1=2, 2=3 y 1=3).
   $$\mu(\pi) = (-1)^{N-|\pi|-1} \prod_{B \in \pi} (|B|-1)!$$
3. **Inclusión-Exclusión Interna**: Para una partición dada con $k$ bloques (es decir, $k$ filas únicas efectivas), calculamos cuántas matrices densas existen. Tratamos la matriz comprimida como una matriz de $k \times N$. Aplicamos un PIE (Principio Inclusión-Exclusión) interno sobre esta matriz para asegurar que ninguna columna sea nula (usando el alfabeto de columnas $v^{k-s}-1$, donde $s$ son las filas forzadas a cero)
$$ID = \left( \sum_{\pi \in \Pi_N, |\pi|<N} \mu(\pi) \cdot \text{Dense}(\pi) \right) - (v-1)$$

Restamos $(v-1)$ al final para remover las matrices constantes no nulas (SM), que matemáticamente cumplen con ser densas e idénticas, pero queremos categorizarlas aparte.

Con esto, se puede generalizar el número de matrices que se observan en cada dimensión por tipo, a excepción de LD y NON, que tienen que ser agrupadas dentro del conjunto de matrices densas. El código siguiente muestra los cálculos generalizados, y los resultados corroboran lo obtenido en secciones previas.

In [2]:
import math

# =============================================================================
# 1. HERRAMIENTAS COMBINATORIAS
# =============================================================================

def generate_partitions(collection):
    """Genera particiones de un conjunto (Bell numbers)."""
    if len(collection) == 1:
        yield [collection]
        return
    first = collection[0]
    for smaller in generate_partitions(collection[1:]):
        for n, subset in enumerate(smaller):
            yield smaller[:n] + [[first] + subset]  + smaller[n+1:]
        yield [[first]] + smaller

def calculate_partition_coefficient(partition, N):
    """Coeficiente de Inclusión-Exclusión (Möbius) para el retículo."""
    k = len(partition)
    sign = (-1)**(N - k - 1)
    block_factor = 1
    for block in partition:
        block_factor *= math.factorial(len(block) - 1)
    return sign * block_factor

# =============================================================================
# 2. FÓRMULAS GENERALES EXACTAS
# =============================================================================

def calculate_exact_distribution(max_n):
    print(f"{'='*80}")
    print(f" ESTIMACIÓN MATRICIAL GENERAL (Alineada con Fuerza Bruta N=1..3)")
    print(f"{'='*80}")

    for N in range(1, max_n + 1):
        v = N + 1
        T_total = v**(N*N)
        
        # --- 1. SM (Scalar Matrix) ---
        SM = v if N > 1 else 1

        # --- 2. CEROS (ZRC, ZR, ZC) ---

        K_dense = 0
        for k in range(N + 1):
            term = math.comb(N, k) * ((-1)**k) * ((v**(N-k) - 1)**N)
            K_dense += term
            
        # Conjuntos de Ceros
        Rows_Zero = T_total - ((v**N - 1)**N) # Total - (Filas No Cero)^N
        Cols_Zero = Rows_Zero # Simetría
        
        # Intersección y Unión
        # Union(ZR, ZC) = Total - K_dense
        # Intersection(ZR, ZC) = |ZR| + |ZC| - Union
        
        Union_Z = T_total - K_dense
        Inter_Z = 2 * Rows_Zero - Union_Z
        
        ZRC = Inter_Z - 1      # Restamos la matriz nula (contada en SM)
        ZR = Rows_Zero - Inter_Z
        ZC = Cols_Zero - Inter_Z

        # --- 3. ID (Identical Rows - Strict Dense) ---
        # Calculamos matrices con filas repetidas dentro del Universo Denso.
        ID_raw = 0
        if N > 1:
            rows_indices = list(range(N))
            for part in generate_partitions(rows_indices):
                k = len(part)
                if k == N: continue # Ignorar filas distintas

                coeff = calculate_partition_coefficient(part, N)
                
                # Inclusión-Exclusión Interna (Dense Universe Logic)
                # Para una matriz comprimida k*N, asegurar No Row Zero y No Col Zero.
                count_dense_struct = 0
                for s in range(k + 1): # s = filas comprimidas forzadas a cero
                    sign = (-1)**s
                    ways_to_zero = math.comb(k, s)
                    
                    # Alfabeto columna efectivo: v^(k-s) - 1 (para no tener col ceros)
                    # Pero debemos elevar a la N (N columnas)
                    eff_alphabet = (v**(k - s)) - 1
                    
                    if eff_alphabet <= 0:
                        term = 0
                    else:
                        term = eff_alphabet**N
                    
                    count_dense_struct += sign * ways_to_zero * term
                
                ID_raw += coeff * count_dense_struct
        
        # (Solo si N > 1, si N=1 ID es 0).
        sm_correction = (v - 1) if N > 1 else 0
        ID = ID_raw - sm_correction
        if ID < 0: ID = 0

        # --- 4. RESTO (LD + NON) ---
        # Todo lo que no es fórmula estructural
        clasificado = SM + ZRC + ZR + ZC + ID
        resto = T_total - clasificado

        # --- REPORTE ---
        print(f"\n>>> DIMENSIÓN N={N} (Total={T_total:,})")
        print(f"{'-'*60}")
        print(f"   SM  (Scalar):             {SM:>15,}")
        print(f"   ZRC (Zero Row & Col):     {ZRC:>15,}")
        print(f"   ZR  (Zero Row Only):      {ZR:>15,}")
        print(f"   ZC  (Zero Col Only):      {ZC:>15,}")
        print(f"   ID  (Identical R/C):      {ID:>15,}")
        print(f"{'-'*60}")
        print(f"   SUMA CLASIFICADA:         {clasificado:>15,}")
        print(f"   RESTO (LD + NON):         {resto:>15,}  <-- RESULTADO FINAL")
        print(f"{'='*60}")
        
        # Verificación automática
        if N == 1: check_vals(N, resto, 1)
        if N == 2: check_vals(N, resto, 2 + 50) # LD+NON = 52
        if N == 3: check_vals(N, resto, 17196 + 212898) # LD+NON = 230094
        if N == 3: check_vals(N, ID, 9906, "ID") # Check específico ID

def check_vals(N, val, expected, label="Resto"):
    res = "CORRECTO" if val == expected else f"FALLO (Esp: {expected})"
    print(f"   [Verificación N={N} {label}]: {res}")

# Ejecutar hasta N=7
calculate_exact_distribution(7)

 ESTIMACIÓN MATRICIAL GENERAL (Alineada con Fuerza Bruta N=1..3)

>>> DIMENSIÓN N=1 (Total=2)
------------------------------------------------------------
   SM  (Scalar):                           1
   ZRC (Zero Row & Col):                   0
   ZR  (Zero Row Only):                    0
   ZC  (Zero Col Only):                    0
   ID  (Identical R/C):                    0
------------------------------------------------------------
   SUMA CLASIFICADA:                       1
   RESTO (LD + NON):                       1  <-- RESULTADO FINAL
   [Verificación N=1 Resto]: CORRECTO

>>> DIMENSIÓN N=2 (Total=81)
------------------------------------------------------------
   SM  (Scalar):                           3
   ZRC (Zero Row & Col):                   8
   ZR  (Zero Row Only):                    8
   ZC  (Zero Col Only):                    8
   ID  (Identical R/C):                    2
------------------------------------------------------------
   SUMA CLASIFICADA:             